In [0]:
SELECT  COUNT(*),
        COUNT(DISTINCT customerID)
FROM    workspace.default.train;

Variáveis Categóricas x Churn

In [0]:
SELECT  LOWER(gender) AS gender,
        SeniorCitizen,
        LOWER(Partner) AS Partner,
        LOWER(Dependents) AS Dependents,
        IF(LOWER(tenure) = 'unknown', 'unknown', LPAD(LOWER(tenure), 2, '0')) AS tenure,
        LOWER(PhoneService) AS PhoneService,
        LOWER(MultipleLines) AS MultipleLines,
        LOWER(InternetService) AS InternetService,
        LOWER(OnlineSecurity) AS OnlineSecurity,
        LOWER(OnlineBackup) AS OnlineBackup,
        LOWER(DeviceProtection) AS DeviceProtection,
        LOWER(TechSupport) AS TechSupport,
        LOWER(StreamingTV) AS StreamingTV,
        LOWER(StreamingMovies) AS StreamingMovies,
        LOWER(Contract) AS Contract,
        LOWER(PaperlessBilling) AS PaperlessBilling,
        LOWER(PaymentMethod) AS PaymentMethod,
        Churn,
        COUNT(*) AS qtde
FROM    workspace.default.train
GROUP BY ALL;

/*
Variáveis com maior variação em relação ao Churn:
- InternetServices (e variáveis relacionadas)
- Contract
- Payment Method
- tenure
*/


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

... (7 outputs left out)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
SELECT  LOWER(InternetService) AS InternetService,
        LOWER(Contract) AS Contract,
        LOWER(PaymentMethod) AS PaymentMethod,
        LOWER(PhoneService) AS PhoneService,
        IF(LOWER(tenure) = 'unknown', 'unknown', LPAD(LOWER(tenure), 2, '0')) AS tenure,
        Churn,
        COUNT(*) AS qtde
FROM    workspace.default.train
GROUP BY ALL;

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
SELECT  MonthlyCharges, TotalCharges
FROM    workspace.default.train
WHERE   LOWER(tenure) = 'unknown';


# Variáveis numéricas

In [0]:
SELECT  MonthlyCharges,
        TotalCharges,
        Churn,
        MonthlyIncome,
        IF(LOWER(tenure) <> 'unknown', CAST(tenure AS INT), NULL) AS tenure
FROM    workspace.default.train;

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

# Variável textual - ClientFeedback

In [0]:
SELECT  Churn,
        CustomerFeedback
FROM    workspace.default.train;

In [0]:
SELECT  customerID,
        ai_classify(
            CustomerFeedback,
            '{
                "preco_alto": "Cliente acha o custo mensal alto ou caro em relação ao mercado, menciona preço mensal, cobranças, orçamento",
                "qualidade_servico": "Quedas frequentes, lentidão, serviço de internet não confiável ou que não atende expectativas",
                "flexibilidade_contrato": "Insatisfação com o tipo/vínculo do contrato, queria contrato mais flexível ou sem fidelidade",
                "metodo_pagamento": "Insatisfação com o método de pagamento disponível (cheque eletrônico, cartão, débito automático)",
                "financeiro_pessoal": "Problemas financeiros pessoais, mudança de endereço ou motivos pessoais alheios à empresa",
                "suporte_atendimento": "Insatisfação com atendimento, suporte técnico ou customer service",
                "satisfeito": "O cliente está satisfeito com o serviço, valor do serviço, qualidade do serviço, não tem reclamações."
            }',
/*            '{
                "preco_alto": "Cliente acha o custo mensal alto ou caro em relação ao mercado, menciona preço mensal, cobranças, orçamento",
                "oferta_concorrente": "Cliente trocou porque encontrou oferta/promoção melhor ou mais barata em outro provedor",
                "qualidade_servico": "Quedas frequentes, lentidão, serviço de internet não confiável ou que não atende expectativas",
                "flexibilidade_contrato": "Insatisfação com o tipo/vínculo do contrato, queria contrato mais flexível ou sem fidelidade",
                "metodo_pagamento": "Insatisfação com o método de pagamento disponível (cheque eletrônico, cartão, débito automático)",
                "financeiro_pessoal": "Problemas financeiros pessoais, mudança de endereço ou motivos pessoais alheios à empresa",
                "suporte_atendimento": "Insatisfação com atendimento, suporte técnico ou customer service"

            }',
*/
            MAP(
                'version', '2.1',
                'instructions', 'Textos são feedbacks em inglês de clientes. Alguns deles cancelaram o serviço de internet e/ou telefone. Se o feedback for de satisfação com o serviço, classifique-o dessa forma. Caso contrário, classifique o MOTIVO PRIMÁRIO da insatisfação. Se, no texto, o clinte indicar que optou por cancelar ou mudar para outra empresa, classifique o MOTIVO PRIMÁRIO dessa decisão.'
            )
        ):response[0].value::STRING AS motivo,
        Churn,
        CustomerFeedback
FROM    workspace.default.train
--WHERE   LOWER(Churn) = 'yes'
LIMIT 50;

## Motivos de Churn — Classificação do CustomerFeedback

Os 1.504 feedbacks de clientes com `Churn = 'Yes'` são textos livres e todos distintos, então agregar diretamente por `CustomerFeedback` não funciona. A análise abaixo usa `ai_classify()` para extrair o motivo primário de cancelamento de cada texto e depois agrega por categoria.

> Cada execução das células abaixo reprocessa os 1.504 textos com IA (pode levar alguns minutos) — evite rodar repetidas vezes.

In [0]:
WITH classificados AS (
    SELECT  customerID,
            ai_classify(
                CustomerFeedback,
                '{
                  "preco_alto": "Cliente acha o custo mensal alto ou caro em relação ao mercado, menciona preço mensal, cobranças, orçamento",
                  "oferta_concorrente": "Cliente trocou porque encontrou oferta/promoção melhor ou mais barata em outro provedor",
                  "qualidade_servico": "Quedas frequentes, lentidão, serviço de internet não confiável ou que não atende expectativas",
                  "flexibilidade_contrato": "Insatisfação com o tipo/vínculo do contrato, queria contrato mais flexível ou sem fidelidade",
                  "metodo_pagamento": "Insatisfação com o método de pagamento disponível (cheque eletrônico, cartão, débito automático)",
                  "financeiro_pessoal": "Problemas financeiros pessoais, mudança de endereço ou motivos pessoais alheios à empresa",
                  "suporte_atendimento": "Insatisfação com atendimento, suporte técnico ou customer service"
                }',
                MAP(
                    'version', '2.1',
                    'instructions', 'Textos são feedbacks em inglês de clientes que cancelaram o serviço de internet e/ou telefone. Classifique o MOTIVO PRIMÁRIO do cancelamento. Se o cliente menciona preço alto e também concorrência, escolha a causa raiz mais determinante declarada por ele.'
                )
            ):response[0].value::STRING AS motivo
    FROM    workspace.default.train
    WHERE   LOWER(Churn) = 'yes'
)
SELECT  motivo,
        COUNT(*) AS qtde,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM    classificados
GROUP BY motivo
ORDER BY qtde DESC;

In [0]:
WITH classificados AS (
    SELECT  customerID,
            ai_classify(
                CustomerFeedback,
                '{
                  "preco_alto": "Cliente acha o custo mensal alto ou caro em relação ao mercado, menciona preço mensal, cobranças, orçamento",
                  "oferta_concorrente": "Cliente trocou porque encontrou oferta/promoção melhor ou mais barata em outro provedor",
                  "qualidade_servico": "Quedas frequentes, lentidão, serviço de internet não confiável ou que não atende expectativas",
                  "flexibilidade_contrato": "Insatisfação com o tipo/vínculo do contrato, queria contrato mais flexível ou sem fidelidade",
                  "metodo_pagamento": "Insatisfação com o método de pagamento disponível (cheque eletrônico, cartão, débito automático)",
                  "financeiro_pessoal": "Problemas financeiros pessoais, mudança de endereço ou motivos pessoais alheios à empresa",
                  "suporte_atendimento": "Insatisfação com atendimento, suporte técnico ou customer service"
                }',
                MAP(
                    'version', '2.1',
                    'instructions', 'Textos são feedbacks em inglês de clientes que cancelaram o serviço de internet. Classifique o MOTIVO PRIMÁRIO do cancelamento. Se o cliente menciona preço alto e também concorrência, escolha a causa raiz mais determinante declarada por ele.'
                )
            ):response[0].value::STRING AS motivo
    FROM    workspace.default.train
    WHERE   LOWER(Churn) = 'yes'
)
SELECT  c.motivo,
        LOWER(t.Contract) AS Contract,
        LOWER(t.InternetService) AS InternetService,
        COUNT(*) AS qtde
FROM    classificados c
JOIN    workspace.default.train t
  ON    t.customerID = c.customerID
GROUP BY ALL
ORDER BY qtde DESC;

In [0]:
SELECT *
FROM workspace.default.train

In [0]:
SELECT *
FROM workspace.default.train_classificado